In [21]:
# -*- coding: utf-8 -*-

# Sample Python code for youtube.channels.list
# See instructions for running these code samples locally:
# https://developers.google.com/explorer-help/code-samples#python

import os

import google_auth_oauthlib.flow
import googleapiclient.discovery
import googleapiclient.errors
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
import google.auth.exceptions
from google.auth.transport.requests import Request

import pickle
from convenient_pickle import *
import time
import datetime
import re

import country_converter as coco
from airports import airport_data
import pycountry

import datetime
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [22]:
#You'll need to change your data path to get this to work
data_path = '[insert your data path here]'
current_directory = os.getcwd()
scopes = ["https://www.googleapis.com/auth/youtube.force-ssl"]

In [23]:
def handle_credentials():
    os.environ["OAUTHLIB_INSECURE_TRANSPORT"] = "1"
    quota = 0
    api_service_name = "youtube"
    api_version = "v3"
    # Change the below to whatever your secrets file is called.
    client_secrets_file = "[your_secrets_file]"
    credentials = None
    if os.path.exists('token.json'):
        try:
            credentials = Credentials.from_authorized_user_file('token.json',scopes)
            credentials.refresh(Request())
        except google.auth.exceptions.RefreshError as error: 
            credentials = None
            print(f'{error}')
    if not credentials or not credentials.valid: 
        if credentials and credentials.expired and credentials.refresh_token: 
            credentials.refresh(Request())
        else: 
            flow = google_auth_oauthlib.flow.InstalledAppFlow.from_client_secrets_file(
                client_secrets_file, scopes)
            credentials = flow.run_local_server(port=0)
    with open('token.json', 'w') as token: 
        token.write(credentials.to_json())
    youtube = googleapiclient.discovery.build(
        api_service_name, api_version, credentials=credentials
        )
    return youtube

In [24]:
def get_video_urls(indict, verbose=False): 
    outdict = dict()
    for key in indict.keys(): 
        url_list = []
        for page in indict[key]: 
            for item in page['items']: 
                youtube_id = item['id']
                if 'videoId' in youtube_id.keys(): 
                    url_list.append(youtube_id['videoId'])
                else: 
                    if verbose:
                        print(youtube_id.keys())
        outdict[key] = url_list
    return outdict

In [25]:
def collect_video_comments(video_info):
    comment_dict = dict()
    
    youtube = handle_credentials()
    bad_vid = 0
    total_vid = 0
    for country in list(video_info.keys()):
        print(f"Starting {country}")
        country_comments = dict()
        for vid in video_info[country]:
            total_vid +=1
            try:
                vid_id = vid['items'][0]['id']
                request = youtube.commentThreads().list(
                    part='snippet',
                    videoId=vid_id
                )
                comment_result = request.execute()
                country_comments[vid_id] = comment_result
            except: 
                bad_vid += 1
        comment_dict[country] = country_comments
        print(f"Finishing {country}")
        print('------------')
    print(f"There were {bad_vid}/{total_vid} total videos where this didn't work.")
    return comment_dict


In [26]:
os.listdir(os.getcwd() + '/country_pickle_files/')

['.ipynb_checkpoints',
 'video_descriptions',
 'video_descriptions_for_Albania.pkl',
 'video_descriptions_for_Austria.pkl',
 'video_descriptions_for_Azerbaijan.pkl',
 'video_descriptions_for_Belgium.pkl',
 'video_descriptions_for_Bosnia.pkl',
 'video_descriptions_for_Croatia.pkl',
 'video_descriptions_for_Georgia.pkl',
 'video_descriptions_for_Germany.pkl',
 'video_descriptions_for_Iceland.pkl',
 'video_descriptions_for_Luxembourg.pkl',
 'video_descriptions_for_Macedonia.pkl',
 'video_descriptions_for_Malta.pkl',
 'video_descriptions_for_Moldova.pkl',
 'video_descriptions_for_Montenegro.pkl',
 'video_descriptions_for_Serbia.pkl',
 'video_descriptions_for_Switzerland.pkl',
 'youtube_travel_top_100_Albania.pkl',
 'youtube_travel_top_100_Austria.pkl',
 'youtube_travel_top_100_Azerbaijan.pkl',
 'youtube_travel_top_100_Belgium.pkl',
 'youtube_travel_top_100_Bosnia.pkl',
 'youtube_travel_top_100_Croatia.pkl',
 'youtube_travel_top_100_Georgia.pkl',
 'youtube_travel_top_100_Germany.pkl',
 'you

In [27]:

currentdir = os.getcwd()

country_dict = dict()

file_list = os.listdir(os.getcwd() + '/country_pickle_files/')

for country_file in [i for i in file_list if "video_descriptions_for" in i]:
    country_name = country_file.split('.')[0].split('_')[-1]
    test = load_pickle(os.getcwd()+'/country_pickle_files/'+country_file)
    country_dict[country_name] = test
    os.chdir(currentdir)

In [28]:
country_file.split('.')[0].split('_')[-1]

'Switzerland'

In [29]:
total_len = 0

for key in country_dict.keys():
    total_len += len(country_dict[key])
print(total_len)
country_dict.keys()

10360


dict_keys(['Albania', 'Austria', 'Azerbaijan', 'Belgium', 'Bosnia', 'Croatia', 'Georgia', 'Germany', 'Iceland', 'Luxembourg', 'Macedonia', 'Malta', 'Moldova', 'Montenegro', 'Serbia', 'Switzerland'])

In [31]:
# country_dict = {i:country_dict[i] for i in country_dict.keys()}

#In the event that you want to just collect the comment threads for a single country, feel free to use the line below

albania = ['Albania']
albania_info = {i: country_dict[i] for i in albania}

In [33]:
#comment_dict = collect_video_comments(country_dict)
comment_dict = collect_video_comments(albania_info)

Starting Albania
Finishing Albania
------------
There were 20/660 total videos where this didn't work.


In [28]:
currentdir = os.getcwd()
for key in comment_dict.keys(): 
    use_dir = os.getcwd() + '/country_pickle_files/video_comments/'
    dump_pickle(use_dir, f"{key}_comments.pkl",comment_dict[key])
    os.chdir(currentdir)